In [16]:
import importlib
import json
from pathlib import Path
from tests.test_minio import test_minio_auth_basic
from cisei_lib.core.profiles.geo_info import DSFlags, P2PLink
from geopy.distance import distance
import cisei_lib.core.rf.rf_engine as rf
import cisei_lib.core.rf.rssi_core as rc
import cisei_lib.core.profiles.geo_info_vector as giv
import cisei_lib.dem.dem_utils as du


# os.environ["MINIO_ENDPOINT"] = "https://s3.ciseiplan.netiswork.pro.br"
#test_minio_auth_basic()
# Start at your current Jupyter notebook location
current_dir = Path.cwd()
print(f"Current Directory: {current_dir}")

with open(current_dir.parent / 'data' / 'worst_eval.json', "r", encoding="utf-8") as f:
    worst = json.load(f)
    
print(worst[0].keys())

print(len(worst))

records = rc.load_json(current_dir.parent / 'data' / 'all_chunks_2.json')
print(worst[0]['record']['tx']['name'])

for k,v in records[0]['features'].items():
    print(k,v)


rssi_model = rc.ExpressionModel.load(current_dir.parent / 'data' / 'final_model.json')
print( rssi_model.learned_coeffs )
rssi_model.predict(worst[0]['record'])

Current Directory: /workspaces/planning_service/tests/jupyter
dict_keys(['idx', 'measured', 'predicted', 'residual', 'abs_error', 'record', 'explain'])
50
SOC-S-GE-018
fspl 96.38
dist_m 1137.63
delta_diffra 0
terrain {'core': 0, 'fresnel': 0, 'boundary': 0}
vegetation {'core': 0, 'fresnel': 0, 'boundary': 0}
buildings {'core': 0, 'fresnel': 55.61, 'boundary': 263.71}
terrain_peaks_vv []
max_obstruction_angle_rad 0
tx_rx_elevation_angle_rad 0.04
tx_near_terminal_clearance_m 8.96
rx_near_terminal_clearance_m 6.38
{'veg_boundary': 2.1122170068771817, 'veg_fresnel': 2.297702928544838, 'veg_core': 1.513868602186462, 'bldg_boundary': 1.247457660132967, 'bldg_fresnel': 0.9631262600057635, 'bldg_core': 1.088907025812846}


-47.008133255464934

In [17]:
importlib.reload(rf)


def find_record(id):
    rx_name = worst[id]['record']['rx']['name']
    tx_name = worst[id]['record']['tx']['name']

    found = None
    for idx, r in enumerate(records):
        if r['tx']['name'] == tx_name and r['rx']['name'] == rx_name:
            found = idx
            break

    return found    
        
def define_link(id):
    link = worst[id]['record']['rx']
    rx, rx_ha = link['pos'], link['ant_height']
    link = worst[id]['record']['tx']
    tx, tx_ha = link['pos'], link['ant_height']
    
    D = distance(tx,rx).meters
    if int(tx[0]) != int(rx[0]) or int(tx[1]) != int(rx[1]):
        print('Links is cross-tile')
    if D < 100:
        print(f'link {id} is too short: {D}')
        return None
    if rx_ha < 7:        
        print(f"link {id} rx_ha adjusted from {rx_ha}")
        rx_ha = 7
    if tx_ha < 7:        
        print(f"link {id} rx_ha adjusted from {tx_ha}")
        tx_ha = 7

    return P2PLink(tx, rx, tx_ha, rx_ha, 900)
       

if False:
    for id in range(len(worst)):            
        print(define_link(id))

rfo = rf.RFEngine()
    
for id in range(len(worst)):
    link = define_link(id)
    idx= find_record(id)

    if link is None or idx is None:
        print(f'What is wrong edgard A {id}')
        continue

    if print(worst[id]['record']['features'] != records[idx]['features']):
        print(f'What is wrong edgard B {id}')
        continue   
    

    rfo.load_profile(link)
    new_features = rfo.evaluate_link()       
    record = records[idx]
    record['features'] = new_features

    print(f'id = {id}')
    print(worst[id]['measured'])
    print(worst[id]['predicted'])
    print(rssi_model.predict(record))






False
Cached building: True
id = 0
-101.0
-47.008133255464934
-47.008133255464934
False
Cached building: True
id = 1
-53.3
-101.85526211092687
-74.01526211092687
False
Cached building: True
id = 2
-60.0
-108.32674553578228
-78.90674553578228
False
Cached building: True
id = 3
-67.1
-111.59202406177008
-110.86202406177007
False
Cached building: True
id = 4
-65.0
-109.27551213388988
-85.25551213388988
False
Cached building: True
id = 5
-69.6
-113.79245431497385
-112.19245431497384
False
Cached building: True
id = 6
-92.1
-48.84619232416272
-48.84619232416272
Links is cross-tile
link 7 rx_ha adjusted from 0.0
False
Cached building: True
id = 7
-74.3
-116.91061362700121
-94.35061362700121
link 8 rx_ha adjusted from 0.0
False
Cached building: True
id = 8
-54.9
-96.92622857263468
-97.15622857263467
False
Cached building: True
id = 9
-70.9
-112.7195141194693
-112.2095141194693
False
Cached building: True
id = 10
-58.7
-100.27733429812317
-98.76733429812316
False
Cached building: True
id = 11
